##olist eccomerce: Data exploration with DuckDb

#1, Setup notebook

### in this notebook we load five tables of the e-commerce dataset into DuckDb and
##    them with SQL before designing a data warehouse.

**what we will do**
1. Load the CSV file in the Duckdb
2. unserstand each table: structure, grain. Keys
3. check the relationships( foreigns)
4. Find data quality issues
5. Build a star schema(data warehouse)

**Tables** Customers,orders,order_items, products, sellers



In [1]:
from pathlib import Path
import duckdb
import pandas as pd


In [9]:
Root = Path.cwd()
Data_dict = Root/ "Data" 
Raw_dir = Data_dict/ "Dataset"
Warehouse_dir = Data_dict/ "Warehouse"
csv = Raw_dir /"olist_orders_dataset.csv"
print(csv,csv.exists())


/home/estifanos/data_mining_class/kddm_lab/Data/Dataset/olist_orders_dataset.csv True


In [10]:
print(duckdb.sql(f"SELECT count(*) FROM '{csv}'"))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        99441 │
└──────────────┘



In [ ]:
### DuckDb lets query CSV data with SQL. The database file persists locally so later notebooks

In [11]:
DB_PATH = Warehouse_dir/ "ecommerce.duckdb"
con = duckdb.connect(str(DB_PATH))

In [12]:
def q(sql: str) -> pd.DataFrame:
    """ RUn a sql query in Duckdb and return a pandas dataframe."""
    return con.sql(sql_str)


## 2, Load the source tables into a **raw schema**

A schema is just a namespace that groups tables. We keep the source copies in **Dataset**
so that they remain distinguishablew from cleaned warehouse tables we will create later

In [13]:
RAW_TABLES: dict[str, dict[str,str]] = {
    "customers" : {"customer_zip_code_prefix": "VARCHAR"}, # keep leading zeroes
    "orders" : {},
    "order_items" : {},
    "products":{},
    "sellers":{},
}

RAW_SCHEMA = "raw"

In [14]:
def csv_path(table: str) -> path:
    return Raw_dir/ f"olist_{table}_dataset.csv"

def check(tables) -> None:
    """failed o nce, list everymissing files"""
    missing = [csv_path(t) for t in tables if not csv_path(t).exists()]
    if missing:
        names = "\n".join(f" - {p}" for p in missing)
        raise FileNotFoundError(f"missing CSv file:{names}")

In [22]:
def load(con: duckdb.DuckDBPyConnection, table: str, column_types: dict[str, str]) -> int:
    target = f'{RAW_SCHEMA}."{table}"'
    source = con.read_csv(str(csv_path(table)), header=True, dtype=column_types or None)
    con.execute(f"DROP TABLE IF EXISTS {target}")
    source.to_table(target)
    return con.execute(f"SELECT count(*) FROM {target}").fetchone()[0]

In [23]:
check(RAW_TABLES)

In [24]:
con.execute(f"CREATE SCHEMA IF NOT EXISTS {RAW_SCHEMA}")
row_counts = {t: load(con, t, types) for t, types in RAW_TABLES.items()}

In [26]:
pd.DataFrame(row_counts.items(), columns = ["table_name","row_counts"])

,table_name,row_counts
0,customers,99441
1,orders,99441
2,order_items,112650
3,products,32951
4,sellers,3095


doing the assigment